# Lecture: Interacting with SQLite Databases using Pandas & SQLAlchemy

create_engine("sqlite:///../datasets/customer_db.db"): Creates a connection object to a SQLite database file located inside the datasets directory.

pd.read_csv("../datasets/customer.csv"): Uses ../ relative path to navigate one directory up from the notebook folder into datasets and load CSV files.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///../datasets/customer_db.db")

customers_df = pd.read_csv("../datasets/customer.csv")
orders_df = pd.read_csv("../datasets/order.csv")

Export Pandas DataFrames directly into database tables inside SQLite using to_sql().

name: Defines the target table name in the database ("customers", "orders").

con=engine: Passes the active database connection.

if_exists="replace": Drops the table if it already exists and creates a new one.

index=False: Prevents Pandas from writing dataframe row numbers (0, 1, 2...) as a column in the database.

In [2]:
# Convert DataFrames to SQL tables
customers_df.to_sql("customers", con=engine, if_exists="replace", index=False)
orders_df.to_sql("orders", con=engine, if_exists="replace", index=False)

3

Retrieve entire tables or selected columns from SQLite directly into a DataFrame using read_sql_table().

pd.read_sql_table("customers", engine): Loads the full customers table.

columns=["name", "phone_number"]: Filters and fetches only the specified columns, saving memory.

In [3]:
# Read full table
pd.read_sql_table("customers", engine)

# Read specific columns only
pd.read_sql_table("customers", engine, columns=["name", "phone_number"])

,name,phone_number
0,Donald,7326784567
1,Bill,6573489999
2,Modi,4567895646


Execute raw SQL queries to join multiple tables and load the resulting dataset.

text(...): Wraps raw SQL statements into an executable SQLAlchemy query object.

INNER JOIN: Combines rows from customers and orders where customers.id matches orders.customer_id.

with engine.connect() as conn:: Safe database context manager that automatically opens and closes the connection block.

pd.read_sql_query(): Runs the query and converts the result set directly into a Pandas DataFrame.

In [ ]:
# Define SQL JOIN query
query = text("""
    SELECT customers.name, orders.amount
    FROM customers
    INNER JOIN orders ON customers.id = orders.customer_id
""")
with engine.connect() as conn:
    df1 = pd.read_sql_query(query, con=conn)

df1

,name,amount
0,Donald,120.0
1,Bill,20.0
2,Modi,950.0


Align column names with existing SQL schema and insert new records into an existing table
df.rename(columns=..., inplace=True): Renames dataframe columns so they match the exact column names present in the database table ("Customer Name" $\rightarrow$ "name")
if_exists="append": Adds new rows to the bottom of the existing table without deleting previous data.

In [7]:
df = pd.read_csv("../datasets/customers.csv")
df.rename(columns={"Customer Name": "name", "Customer Phone": "phone_number"}, inplace=True)

df.to_sql(name="customers", con=engine, index=False, if_exists="append")

6

Build a custom DataFrame from scratch using Python dictionaries and write it as a new database table.

pd.DataFrame(...): Constructs a 2-column DataFrame (name and marks).

pd.read_sql('student', con=engine): Generic method to read data by passing a table name or raw SQL string.

In [8]:
student_data = pd.DataFrame({
    'name': ['pari', 'ayush', 'love'],
    'marks': [84, 99, 74]
})

# Save to database
student_data.to_sql('student', engine, if_exists='replace', index=False)

# Read student table
pd.read_sql('student', con=engine)

,name,marks
0,pari,84
1,ayush,99
2,love,74
